# LLM Images — Vision / Multimodal Input

Send an image alongside a text prompt to a vision-capable model. Each provider uses a slightly different format for attaching image data.

Set `IMAGE_FILENAME` in the shared cell to point to any local image. For Ollama, `gemma3:4b` is used — the locally installed vision model.

In [ ]:
from PIL import Image
from IPython.display import display
from io import BytesIO
import base64
import os

IMAGE_FILENAME = 'image.jpg'
IMAGE_PROMPT   = 'Describe this image in detail.'

image = Image.open(IMAGE_FILENAME)
with open(IMAGE_FILENAME, 'rb') as f:
    image_bytes = f.read()
print(f'{IMAGE_FILENAME}: {image.size} px, mode={image.mode}')

display(image)

In [ ]:
def to_base64(img, fmt='JPEG'):
    buf = BytesIO()
    save_img = img.convert('RGB') if img.mode == 'RGBA' and fmt == 'JPEG' else img
    save_img.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    mime = f'image/{fmt.lower()}'
    return b64, mime, f'data:{mime};base64,{b64}'

b64_data, mime_type, data_url = to_base64(image)

---
## OpenAI

Pass the image as a `data:` URL inside an `image_url` content block.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{
        'role': 'user',
        'content': [
            {'type': 'text', 'text': IMAGE_PROMPT},
            {'type': 'image_url', 'image_url': {'url': data_url, 'detail': 'high'}}
        ]
    }]
)
print(openai_resp.choices[0].message.content)

---
## Anthropic

Pass the image as a `base64` source inside an `image` content block.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=500,
    messages=[{
        'role': 'user',
        'content': [
            {
                'type': 'image',
                'source': {
                    'type': 'base64',
                    'media_type': mime_type,
                    'data': b64_data
                }
            },
            {'type': 'text', 'text': IMAGE_PROMPT}
        ]
    }]
)
print(anthropic_resp.content[0].text)

---
## Google Gemini

Pass the PIL Image object directly — Gemini accepts it natively.

In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

genai.configure(api_key=GEMINI_API_KEY)
gemini_vision_model = genai.GenerativeModel(GOOGLE_MODEL)
gemini_resp = gemini_vision_model.generate_content([IMAGE_PROMPT, image])
print(gemini_resp.text)

---
## Ollama (local) — `gemma3:4b`

Pass image bytes in the `images` list of the message. Only vision-capable models support this; `gemma3:4b` is confirmed installed.

In [ ]:
import ollama

OLLAMA_VISION_MODEL = 'gemma3:4b'

ollama_resp = ollama.chat(
    model=OLLAMA_VISION_MODEL,
    messages=[{
        'role': 'user',
        'content': IMAGE_PROMPT,
        'images': [image_bytes]
    }]
)
print(ollama_resp['message']['content'])